# Running cellpose with GPUs

In [1]:
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

from skimage import io
from skimage.filters import threshold_otsu

import zarr
from cellpose import models, core

use_GPU = core.use_gpu(gpu_number=0)
print('>>> GPU activated? %d'%use_GPU)

import ray

from dask import array as da

import mFISHwarp.morphology
import mFISHwarp.utils
import mFISHwarp.zarr

import pandas as pd
from ome_zarr.writer import write_multiscales_metadata

2024-08-21 13:26:56,673	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.6.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


>>> GPU activated? 1


## Load model

In [2]:
# path to dataset and model
dataset_folder = "/mnt/ampa02_data01/tmurakami/model_training/crops_tophat"

train_folder = os.path.join(dataset_folder,'training')
models_path = os.path.join(train_folder,'models')

models_file = os.listdir(models_path); models_file.sort()
model_path = os.path.join(models_path,models_file[-1])

model = models.CellposeModel(gpu=use_GPU, pretrained_model=model_path)

## Load N5 or Zarr

In [3]:
data_path = '/mnt/ampa02_data01/tmurakami/240425_whole_4color_2nd_M037-3pb/fused/fused.n5'

res_analysis = 0 # resolution to be analyzed. usually the highest resolution.
res_mask = 4 # resolution to make mask. this does not need to be high.

# lazily load images using dask
imgs = mFISHwarp.zarr.omezarr_bdn5_to_dask(data_path,resolution=res_analysis)
imgs_mask = mFISHwarp.zarr.omezarr_bdn5_to_dask(data_path,resolution=res_mask)

In [4]:
# decide which channel to analyze
segment_chans = [2,1,4]
reference_chans = [3,3,3]
name_segment_chans = ['Gad1','Npy','Pvalb']
segment_save_dir = None

if segment_save_dir is None:
    segment_save_dir = os.path.join(os.path.dirname(os.path.dirname(data_path)),'segmentation')
    
if len(segment_chans)!=len(reference_chans) or len(segment_chans)!=len(name_segment_chans):
    raise ValueError("please make the lists in the same size")

## Create mask

In [5]:
# downsampling
img_down_ref = imgs_mask[reference_chans[0],...].compute()
global_thresh = threshold_otsu(img_down_ref)
img_mask = mFISHwarp.morphology.mask_maker(img_down_ref,global_thresh)

# import napari
# viewer = napari.Viewer()
# viewer.add_image(img_mask)
# viewer.add_image(img_down_ref)

## Set basic information

In [7]:
### Parameters
auto_diam = False # Cellpose automatic diameter estimation.
# theoretically, anisotropy parameter affects the accuracy. However in practice, changing this values to be the exact voxel ratio does not significantly add accuracy. 
# this may be because of the non-isotropic PSF of light-sheet.
voxel_size = (2.0,1.3,1.3)
anisotropy = voxel_size[1]/voxel_size[0]
min_size = 40

# Channel parameters which were used during the training.
Training_channel = 2 # I do not know why but the cellpose see the images as KRGB. If the color is green, set it to 2.
Second_training_channel = 1

# lazyly read image and convert to dask array
chunk_size = (256,512,512)
depth = (32,64,64) 
boundary = "reflect"

# get basic information of the images and overlapped images for zarr
img_ref = imgs[reference_chans[0],...]
img_ref = da.rechunk(img_ref,chunks=chunk_size)
img_shape = img_ref.shape
img_chunks = mFISHwarp.utils.chunks_from_dask(img_ref)
img_ref_overlap = da.overlap.overlap(img_ref, depth, boundary)
img_overlap_shape = img_ref_overlap.shape
img_overlap_chunks = mFISHwarp.utils.chunks_from_dask(img_ref_overlap)


# If mask is used, calculate which chunks will be segmented
flag_array = mFISHwarp.utils.flag_array_generator(chunk_size, img_ref.shape, img_mask)
print(f'{flag_array.sum()} blocks of {flag_array.shape[0]}*{flag_array.shape[1]}*{flag_array.shape[2]}={flag_array.size} blocks will be calculated')

1454 blocks of 11*17*13=2431 blocks will be calculated


## Prepare zarr container to save segmentation

In [8]:
# make save zarr directories in advance
for name_segment_chan in name_segment_chans:
    ### zarr to save segmentation
    labeled_overlap_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'segmented_overlap.zarr')
    labeled_overlap_zarr = zarr.open(
        labeled_overlap_zarr_path,
        mode='a', 
        shape=img_overlap_shape, 
        chunks=img_overlap_chunks, 
        dtype=np.int32
    )
    # add attribute so that I can know the overlap size later.
    labeled_overlap_zarr.attrs.update({"overlap_size": depth})


    ### zarr to save probability and original image. OME-Zarr for browsing purpose
    prob_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'prob.zarr')
    # create zarr
    store = zarr.DirectoryStore(prob_zarr_path, dimension_separator='/')
    root = zarr.group(store=store)
    # np.int8 may not be appropriate because it may cause clipping, but it works for now to reduce the datasize.
    prob_zarr = root.create_dataset('0',shape=(2,)+img_shape,chunks=(1,)+img_chunks,dtype=np.int8)
    # prepare metadata to zarr
    datasets = mFISHwarp.zarr.datasets_metadata_generator((1.0,)+voxel_size, downscale_factor=(1,1,1,1), pyramid_level=1)
    write_multiscales_metadata(root, datasets=datasets, axes=['c','z','y','x'])
    prob_zarr.attrs.update({"overlap_size": depth})

## Define functions to run in parallel

In [9]:
from cupyx.scipy.ndimage import white_tophat
from skimage.morphology import ball
import cupy as cp

def preprocessing(img, ball_size, norm_values_ref, norm_values_tar):
    """
    Define your preprocessing here. 
    In this specific case, I used tophat fileter and normalization.
    img: (c,z,y,x)
    """
    
    # tophat filter
    img = img.astype(float)
    footprint = ball(ball_size)
    footprint_cu = cp.asarray(footprint)
    res = []
    for i in range(img.shape[0]):
        img_cp = cp.asarray(img[i,...])
        res.append(cp.asnumpy(white_tophat(img_cp,footprint=footprint_cu)))
        del img_cp
    
    del footprint_cu
    cp._default_memory_pool.free_all_blocks()

    # normalization
    img = np.stack([
        mFISHwarp.utils.normalization_two_values(res[0], norm_values_ref[0], norm_values_ref[1]),
        mFISHwarp.utils.normalization_two_values(res[1], norm_values_tar[0], norm_values_tar[1])
    ])
    
    
    return img

In [10]:
@ray.remote(num_gpus=0.5,max_calls=1)
def segmentor(
    chunks, # list of images. reference and target
    channels,
    model,
    anisotropy,
    index,
    min_size,
    zarr_file,
    prob_zarr,
    *args,
    **kwargs
):
    # convert dask array to numpy array.
    chunks = np.stack([i.compute() for i in chunks])
    
    # do your defined preprocessing if necessary.
    chunks = preprocessing(chunks,*args,**kwargs)
    
    # run cellpose
    segments, flow, _  = model.eval(chunks, channels=channels, normalize=False, z_axis=1, diameter=model.diam_mean, do_3D=True, min_size=min_size, progress=False, anisotropy=anisotropy, tile=False)
    segments = segments.astype(np.int32)
    prob = flow[2]
    
    # save masks to zarr
    chunk_info = da.from_zarr(zarr_file).chunks
    zarr_file[mFISHwarp.utils.obtain_chunk_slicer(chunk_info, index)] = segments
    
    # save probability to zarr
    if prob_zarr is not None:
        prob_chunk_info = da.from_zarr(prob_zarr).chunks[1:]
        overlap_size = prob_zarr.attrs['overlap_size']
        shape = [i[j] for i, j in zip(prob_chunk_info, index)]
        slicer = tuple(slice(i, i + j) for i, j in zip(overlap_size, shape))
        
        # following is specific to the normalizing preprocessing function
        norm_img = np.clip(chunks[-1][slicer]*255-128,-127,127)  
        int_prob = np.clip(prob[slicer],-127,127)
        
        # save to zarr
        prob_zarr[(0,)+mFISHwarp.utils.obtain_chunk_slicer(prob_chunk_info, index)] = norm_img.astype(np.int8)
        prob_zarr[(1,)+mFISHwarp.utils.obtain_chunk_slicer(prob_chunk_info, index)] = int_prob.astype(np.int8)

In [11]:
for i, segment_chan in enumerate(segment_chans):

    reference_chan = reference_chans[i]
    segment_chan = segment_chans[i]
    name_segment_chan = name_segment_chans[i]

    # make and lazily load overlaped images
    overlap_imgs = []
    img_ref = imgs[reference_chan,...]
    img_ref = da.rechunk(img_ref,chunks=chunk_size)
    overlap_imgs.append(da.overlap.overlap(img_ref, depth, boundary))
    # target
    img = imgs[segment_chan,...]
    img = da.rechunk(img,chunks=chunk_size)
    overlap_imgs.append(da.overlap.overlap(img, depth, boundary))

    # open zarr files
    labeled_overlap_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'segmented_overlap.zarr')
    prob_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'prob.zarr')
    labeled_overlap_zarr = zarr.open(labeled_overlap_zarr_path, mode='a')
    prob_zarr = zarr.open(os.path.join(prob_zarr_path,'0'), mode='a')

    # get files in the zarr in case resume from the middle of the computation
    stored_chunks = os.listdir(labeled_overlap_zarr_path)
    stored_chunks.sort()

    idxs = mFISHwarp.utils.get_dask_index(overlap_imgs[0])

    diameter_yx = model.diam_mean
    anisotropy = anisotropy

    min_size = 40
    model_type = model
    channels = [Training_channel, Second_training_channel]

    # parameters for pre-processing
    ball_size = 10
    normalization_metadata = "/mnt/ampa02_data01/tmurakami/model_training/norm_values_tophat.pickle"
    data_path = '/mnt/ampa02_data01/tmurakami/240425_whole_4color_2nd_M037-3pb/fused/top_hat_10.zarr'
    norm_info = pd.read_pickle(normalization_metadata)
    norm_values_ref = [norm_info[data_path][reference_chan]['lower'], norm_info[data_path][reference_chan]['upper']]
    norm_values_tar = [norm_info[data_path][segment_chan]['lower'], norm_info[data_path][segment_chan]['upper']]

    for index in idxs:
        if flag_array[index[0],index[1],index[2]]:
            if '.'.join([str(i) for i in index]) not in stored_chunks:
                if index[0] == 5:
                    input_blocks = [mFISHwarp.utils.slicing_with_chunkidx(img, index) for img in overlap_imgs]
                    segmentor.remote(
                        input_blocks,
                        [Training_channel, Second_training_channel],
                        model,
                        anisotropy,
                        index,
                        min_size,
                        labeled_overlap_zarr,
                        prob_zarr,
                        ball_size,
                        norm_values_ref,
                        norm_values_tar
                    )

2024-08-21 13:29:03,115	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
